In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/2025.csv", nrows=5)

df.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_Price,Max_Price,Modal_Price,Commodity_Code
0,Andhra Pradesh,Chittor,Chittoor,Gur (Jaggery),NO 2,FAQ,2025-01-01,3200.0,3500.0,3500.0,74
1,Tamil Nadu,The Nilgiris,Gudalur (Uzhavar Sandhai ),Garlic,Average,Local,2025-01-01,28000.0,30000.0,30000.0,25
2,Tamil Nadu,The Nilgiris,Gudalur (Uzhavar Sandhai ),Ginger (Green),Green Ginger,Local,2025-01-01,7000.0,8000.0,8000.0,103
3,Tamil Nadu,The Nilgiris,Gudalur (Uzhavar Sandhai ),Green Chilli,Green Chilly,Local,2025-01-01,4500.0,5000.0,5000.0,87
4,Tamil Nadu,The Nilgiris,Gudalur (Uzhavar Sandhai ),Mint (Pudina),Mint (Pudina),Local,2025-01-01,4500.0,5000.0,5000.0,360


In [2]:
df.shape

(5, 11)

In [3]:
df.columns

Index(['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade',
       'Arrival_Date', 'Min_Price', 'Max_Price', 'Modal_Price',
       'Commodity_Code'],
      dtype='str')

In [4]:
df["Commodity"].unique()

<ArrowStringArray>
['Gur (Jaggery)', 'Garlic', 'Ginger (Green)', 'Green Chilli', 'Mint (Pudina)']
Length: 5, dtype: str

In [5]:
df["Commodity"].value_counts()

Commodity
Gur (Jaggery)     1
Garlic            1
Ginger (Green)    1
Green Chilli      1
Mint (Pudina)     1
Name: count, dtype: int64

In [6]:
df[["Min_Price", "Max_Price", "Modal_Price"]].describe()

,Min_Price,Max_Price,Modal_Price
count,5.000000,5.000000,5.000000
mean,9440.000000,10300.000000,10300.000000
std,10466.279186,11133.283433,11133.283433
min,3200.000000,3500.000000,3500.000000
25%,4500.000000,5000.000000,5000.000000
50%,4500.000000,5000.000000,5000.000000
75%,7000.000000,8000.000000,8000.000000
max,28000.000000,30000.000000,30000.000000


In [7]:
df["Arrival_Date"].head()

0    2025-01-01
1    2025-01-01
2    2025-01-01
3    2025-01-01
4    2025-01-01
Name: Arrival_Date, dtype: str

In [8]:
df.dtypes

State                 str
District              str
Market                str
Commodity             str
Variety               str
Grade                 str
Arrival_Date          str
Min_Price         float64
Max_Price         float64
Modal_Price       float64
Commodity_Code      int64
dtype: object

In [9]:
total_rows = 0

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    total_rows += len(chunk)

total_rows

5819482

In [10]:
target_counts = {
    "Tomato": 0,
    "Onion": 0,
    "Potato": 0
}

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    for commodity in target_counts:
        target_counts[commodity] += (chunk["Commodity"] == commodity).sum()

target_counts

{'Tomato': np.int64(210960),
 'Onion': np.int64(220991),
 'Potato': np.int64(204136)}

In [11]:
date_min = None
date_max = None

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    dates = pd.to_datetime(chunk["Arrival_Date"], errors="coerce")

    chunk_min = dates.min()
    chunk_max = dates.max()

    if date_min is None or chunk_min < date_min:
        date_min = chunk_min

    if date_max is None or chunk_max > date_max:
        date_max = chunk_max

print("Earliest date:", date_min)
print("Latest date:", date_max)

Earliest date: 2025-01-01 00:00:00
Latest date: 2025-12-30 00:00:00


In [12]:
target_markets = {
    "Tomato": set(),
    "Onion": set(),
    "Potato": set()
}

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    for commodity in target_markets:
        markets = chunk.loc[
            chunk["Commodity"] == commodity,
            "Market"
        ].dropna().unique()

        target_markets[commodity].update(markets)

for commodity, markets in target_markets.items():
    print(commodity, ":", len(markets), "unique markets")

Tomato : 1914 unique markets
Onion : 1932 unique markets
Potato : 1857 unique markets


In [13]:
target_states = {
    "Tomato": set(),
    "Onion": set(),
    "Potato": set()
}

for chunk in pd.read_csv("../data/raw/2025.csv", chunksize=100_000):
    for commodity in target_states:
        states = chunk.loc[
            chunk["Commodity"] == commodity,
            "State"
        ].dropna().unique()

        target_states[commodity].update(states)

for commodity, states in target_states.items():
    print(commodity, ":", len(states), "unique states")
    

Tomato : 25 unique states
Onion : 28 unique states
Potato : 28 unique states
